# 06b · Modelado: Propuesta B — Seq2Seq BiLSTM con atención + ensamble neuro-estocástico (Kalman)

| | |
|---|---|
| **Fase CRISP-DM** | 4 — Modelado |
| **Objetivo** | Implementar la Propuesta B (4.ª del Benchmarking, 3.80) tal como la describe la literatura, para la comparación. |
| **Entradas** | `data/gold/series_filtradas.parquet` (voltaje real, calidad, covariables; solo train), `data/gold/series_kalman.parquet` (nivel y pendiente de Kalman), `data/gold/escaladores_kalman.json`, `models/kalman_params.json` |
| **Salidas** | `models/modelo_B.pt` (red + α del ensamble), `reports/modelado/b_*` |

**Componentes y fuentes**
- **Red (Gou et al., 2024):** Seq2Seq con encoder **BiLSTM**, decoder LSTM con **atención multi-cabeza** sobre los estados del encoder y **decodificación paso a paso** (cada día se calcula a partir de la predicción del día anterior, hasta 180). Se entrena sin teacher forcing: el decoder se alimenta siempre con su propia predicción, como en inferencia.
- **Filtro de Kalman (Khan et al., 2026):** suaviza la señal de entrada (notebook 05b) y además **genera su propio pronóstico**: `V̂_Kalman(t0 + h) = nivel(t0) + pendiente(t0) · h`.
- **Ensamble neuro-estocástico (Khan et al., 2026):** `V̂ = α · V̂_red + (1 − α) · V̂_Kalman`, con `α ∈ [0,3; 0,7]` elegido en el bloque final de train. El artículo lo justifica porque la red tiene poco sesgo y mucha varianza, y el Kalman lo contrario.

**Tal cual la literatura:** una sola semilla (`SEED`), igual que los demás modelos, y **sin máscara de saltos** (mejora propia de nuestras variantes). Mismo protocolo de early stopping + reentrenamiento. Por el costo del decoder de 180 pasos, los orígenes de train se submuestrean cada 10 días.

In [ ]:
# Configuración: localiza la raíz del proyecto (local o Colab) y agrega src/ al path
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Poc-TP1")
except ImportError:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "voltcast").is_dir())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import json
import time
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from voltcast import config, io
from voltcast import dataset as ds
from voltcast import model as mdl

config.ensure_dirs()
FIG_DIR = config.REPORTS_DIR / "modelado"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CFG = config.SEQ2SEQ
print(f"Dispositivo: {DEVICE} | semilla: {config.SEED} | α candidatos: {config.FUSION_ALPHAS}")
print("Configuración:", CFG)


def savefig(fig, name):
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    print(f"Figura guardada: {path.relative_to(config.BASE_DIR)}")

## 1. Datos de train con la señal de Kalman

In [ ]:
gold = io.read_layer("gold", "series_filtradas.parquet")
kal = io.read_layer("gold", "series_kalman.parquet")
esc = io.read_json("gold", "escaladores_kalman.json")
SIGMA = esc["sigma_delta"]
train = gold[gold["split"] == "train"].drop(columns="v_filtrada")
train["celda"] = train["celda"].astype(str)
kal["celda"] = kal["celda"].astype(str)
train = train.merge(kal[["fecha", "celda", "v_kalman", "pendiente_kalman"]], on=["fecha", "celda"], how="left",
                    validate="one_to_one")
train = train.rename(columns={"v_kalman": "v_filtrada"})  # la clase de tensores usa la columna 'v_filtrada'
CELLS = sorted(train["celda"].unique())

data = ds.SeriesTensors(train, CELLS, SIGMA, DEVICE)
T_ = len(data.dates)
CUT = T_ - config.ES_BLOCK_DAYS
TR_RANGE = (config.LOOKBACK_DAYS - 1, CUT - 1)
VAL_PAIRS = data.origins(CUT, T_ - 2, stride=7)
PEND = torch.tensor(ds.pivot(train, "pendiente_kalman", CELLS).reindex(data.dates).to_numpy(float),
                    dtype=torch.float32, device=DEVICE)
print(f"{len(CELLS)} celdas | σ_Δ (Kalman) = {SIGMA:.4f} V | orígenes usables: {int(data.ok.sum()):,} | "
      f"bloque interno: {len(VAL_PAIRS[0]):,} pares desde {data.dates[CUT].date()}")

## 2. Entrenamiento de la red (semilla única, sin máscara de saltos)

In [ ]:
t = time.time()
torch.manual_seed(config.SEED)
m_es = mdl.build_model(len(CELLS), len(ds.FEATURES), CFG).to(DEVICE)
r_es = mdl.fit(m_es, data, TR_RANGE, CUT - 1, CFG, val_pairs=VAL_PAIRS, val_target_limit=T_ - 1,
               seed=config.SEED, mask_jumps=False, log=print)
t_es = time.time() - t
print(f"→ mejor época {r_es['mejor_epoca']} ({t_es / 60:.1f} min, {mdl.count_params(m_es):,} parámetros)")
t = time.time()
torch.manual_seed(config.SEED)
final = mdl.build_model(len(CELLS), len(ds.FEATURES), CFG).to(DEVICE)
r_fin = mdl.fit(final, data, (config.LOOKBACK_DAYS - 1, T_ - 2), T_ - 1, CFG, epochs=r_es["mejor_epoca"],
                seed=config.SEED, mask_jumps=False, log=None)
t_fin = time.time() - t
print(f"→ reentrenamiento con todo train: {t_fin / 60:.1f} min")

## 3. Ensamble neuro-estocástico: elección de α en el bloque final de train

Para cada par (celda, origen) del bloque interno se calculan el pronóstico de la red (modelo del early stopping, que no vio ese bloque) y el del Kalman, ambos en las unidades normalizadas del objetivo. Se elige el `α` del rango del artículo (0,3–0,7) que minimiza el MSE del ensamble.

In [ ]:
@torch.no_grad()
def pronosticos_bloque(model, c, t, bs=1024):
    """Objetivo, máscara y pronósticos (normalizados) de la red y del Kalman para los pares (c, t)."""
    model.eval()
    Y, M, PR, PK = [], [], [], []
    h = torch.arange(1, config.HORIZON_DAYS + 1, device=DEVICE, dtype=torch.float32)
    for i in range(0, len(t), bs):
        cc, tt = c[i:i + bs], t[i:i + bs]
        x, y, mask, cell, el = data.batch(cc, tt, T_ - 1)
        win = tt[:, None] + data.offs_in
        anc = data.VF[win, cc[:, None]][:, -config.ANCHOR_DAYS:].mean(1)
        nivel, pend = data.VF[tt, cc], torch.nan_to_num(PEND[tt, cc])
        pk = (nivel[:, None] + pend[:, None] * h - anc[:, None]) / SIGMA
        Y.append(y); M.append(mask); PR.append(model(x, cell, el)); PK.append(pk)
    return torch.cat(Y), torch.cat(M), torch.cat(PR), torch.cat(PK)


Yb, Mb, PRb, PKb = pronosticos_bloque(m_es, *VAL_PAIRS)
filas = []
for a in [0.0, 1.0] + config.FUSION_ALPHAS:
    pe = a * PRb + (1 - a) * PKb
    filas.append({"alpha": a, "MSE_norm": float(mdl.masked_mse(pe, Yb, Mb)),
                  "RMSE_V": float(mdl.masked_mse(pe, Yb, Mb).sqrt()) * SIGMA})
fusion = pd.DataFrame(filas)
cand = fusion[fusion["alpha"].isin(config.FUSION_ALPHAS)]
ALPHA = float(cand.loc[cand["MSE_norm"].idxmin(), "alpha"])
fusion["etiqueta"] = fusion["alpha"].map(lambda a: "solo Kalman" if a == 0 else ("solo red" if a == 1 else f"α = {a:.2f}"))
print(f"α elegido = {ALPHA:.2f}")
fusion.set_index("etiqueta")[["MSE_norm", "RMSE_V"]].round(5)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
h = pd.DataFrame(r_es["historia"])
axes[0].plot(h["epoca"], h["train_mse"], label="train")
axes[0].plot(h["epoca"], h["val_mse"], ls="--", label="bloque interno")
axes[0].axvline(r_es["mejor_epoca"], color="grey", ls=":")
axes[0].set(yscale="log", xlabel="época", ylabel="MSE (normalizado)", title="Seq2Seq-Att: curvas de entrenamiento")
axes[0].legend()
for nombre, pred in [("red", PRb), ("Kalman", PKb), (f"ensamble α={ALPHA:.2f}", ALPHA * PRb + (1 - ALPHA) * PKb)]:
    se = ((pred - Yb) ** 2 * Mb).sum(0) / Mb.sum(0)
    axes[1].plot(range(1, config.HORIZON_DAYS + 1), (se.sqrt() * SIGMA).cpu(), label=nombre)
axes[1].set(xlabel="horizonte (días)", ylabel="RMSE (V)", title="Bloque interno: red, Kalman y ensamble")
axes[1].legend()
fig.tight_layout(); savefig(fig, "b_curvas_ensamble")

**Lectura.**
- **Red:** entrena de forma estable; el mejor punto llega en la época 5 (sin máscara de saltos, la red deja de mejorar antes que nuestras variantes, como ocurrió en el notebook 06).
- **Ensamble neuro-estocástico:** en el bloque interno, el pronóstico del Kalman solo (0,0250 V de RMSE) es peor que el de la red sola (0,0204 V), y **ninguna mezcla mejora a la red sola**: el error baja monótonamente al aumentar α. Dentro del rango del artículo (0,3–0,7) el mejor es **α = 0,70**, en el límite superior (0,0206 V).
- La razón es la misma que vimos con BO-VMD y con el filtro: la serie diaria casi no tiene ruido, así que la ventaja del Kalman (baja varianza) no compensa su sesgo (proyecta el nivel con una pendiente casi nula). Se respeta el rango del artículo y se documenta.
- **Costo:** ~13 min de entrenamiento (early stopping + reentrenamiento), por el decoder de 180 pasos.

## 4. Guardado

In [ ]:
kalman_params = json.loads((config.MODELS_DIR / "kalman_params.json").read_text(encoding="utf-8"))
meta = {
    "nombre": "Propuesta B (Seq2Seq-Att + Kalman)", "arch": "seq2seq", "seed": config.SEED, "cells": CELLS,
    "features": ds.FEATURES, "train_cfg": CFG, "lookback": config.LOOKBACK_DAYS, "horizon": config.HORIZON_DAYS,
    "anchor_days": config.ANCHOR_DAYS, "escaladores": esc, "filtro": "kalman", "kalman_params": kalman_params,
    "alpha_fusion": ALPHA, "fusion_bloque_interno": fusion.drop(columns="etiqueta").to_dict(orient="records"),
    "mask_post_jump": False, "refit_full_train": True, "mejor_epoca": r_es["mejor_epoca"],
    "tiempo_min": {"entrenamiento_min": (t_es + t_fin) / 60},
    "entrenado_hasta": str(data.dates[-1].date()), "es_block_desde": str(data.dates[CUT].date()),
    "fecha_entrenamiento": pd.Timestamp.now().isoformat(timespec="seconds"),
}
path = config.MODELS_DIR / "modelo_B.pt"
mdl.save_checkpoint(path, final, meta)
hist = [{"fase": "early_stopping", **x} for x in r_es["historia"]] + [{"fase": "reentrenamiento", **x} for x in r_fin["historia"]]
pd.DataFrame(hist).to_csv(FIG_DIR / "b_historial_entrenamiento.csv", index=False)
fusion.to_csv(FIG_DIR / "b_fusion_alpha.csv", index=False)
print(f"✔ {path.relative_to(config.BASE_DIR)} ({path.stat().st_size / 1e6:.2f} MB) | α = {ALPHA:.2f}")

## 5. Verificaciones

In [ ]:
x, y, mask, cell, el = data.batch(VAL_PAIRS[0][:256], VAL_PAIRS[1][:256], T_ - 1)
m, ck = mdl.load_checkpoint(path, DEVICE)
with torch.no_grad():
    p1, p2 = m(x, cell, el), final.eval()(x, cell, el)
checks = {
    "el checkpoint reproduce las predicciones": torch.allclose(p1, p2, atol=1e-5),
    f"salida [B, {config.HORIZON_DAYS}] finita": p1.shape[1] == config.HORIZON_DAYS and bool(torch.isfinite(p1).all()),
    "α dentro del rango del artículo": 0.3 <= ck["alpha_fusion"] <= 0.7,
    "solo se usaron fechas de train": data.dates[-1] < gold.loc[gold["split"] == "val", "fecha"].min(),
}
for k, v in checks.items():
    print(("✔" if v else "✘"), k)
assert all(checks.values())

## Conclusión

- Se implementó la **Propuesta B** tal como la describe la literatura: Seq2Seq BiLSTM con atención multi-cabeza y decodificación paso a paso (Gou et al., 2024), filtro de Kalman y ensamble neuro-estocástico red + Kalman (Khan et al., 2026), con una sola semilla y sin máscara de saltos.
- En nuestros datos el ensamble con el Kalman no aporta; el peso de la red queda en el máximo permitido (α = 0,70).
- Salida: `models/modelo_B.pt` (red + α). La comparación con los demás modelos está en el notebook 07.